# §12.2.5 — 절단 길이에 따른 장기 의존 과제 성능

> 딥러닝 교재 · 3부 12장 2절 5항 (🐍)
> 선행: §12.2.1(BPTT) · §12.2.2(절단의 편향) · §12.2.6(구조적 상한)

## 이 노트북이 답하는 질문

1. **절단 길이 $K$가 의존 거리 $\Delta$보다 짧으면** 정말 학습이 불가능한가?
2. 실패는 어떤 **모양**으로 오는가 — 요동인가, 조용한 평평함인가?
3. 경계는 $K=\Delta$에서 얼마나 날카로운가?

**예상 실행 시간** CPU 약 2분 30초 (`FAST = True`이면 약 60초).
과제는 의존 거리를 정확히 제어할 수 있는 덧셈 과제(adding problem)다.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 덧셈 과제 — 의존 거리를 손잡이로

길이 $T$의 값 열 $v_t\sim U(0,1)$과 표시 열 $m_t\in\{0,1\}$을 입력한다. 표시는 정확히 두 곳,
**마지막에서 $\Delta$만큼 떨어진 곳**과 마지막 근처에 있다. 목표는 두 표시 위치의 값의 합.
첫 표시를 기억해 $\Delta$스텝 운반해야만 풀리므로, $\Delta$가 의존 거리다.

In [ ]:
T_SEQ = 40; M = 32

def make_adding(n, delta, rn):
    v = rn.uniform(0, 1, (n, T_SEQ))
    m = np.zeros((n, T_SEQ))
    p1 = T_SEQ - 1 - delta
    p2 = T_SEQ - 1 - rn.integers(0, 3, n)          # 마지막 근처
    m[np.arange(n), p1] = 1.0
    m[np.arange(n), p2] = 1.0
    y = v[np.arange(n), p1] + v[np.arange(n), p2]
    X = np.stack([v, m], axis=2)                    # (n,T,2)
    return X, y

---
## 2. 절단 BPTT — 창 경계에서 상태는 넘기고 기울기는 끊는다

In [ ]:
class RNN:
    def __init__(self, rn, m=M, d_in=2):
        self.Wh = rn.standard_normal((m, m)) * (0.95/np.sqrt(m))
        self.Wx = rn.standard_normal((d_in, m)) * (1/np.sqrt(d_in))
        self.b = np.zeros(m)
        self.U = rn.standard_normal(m)/np.sqrt(m); self.c = np.zeros(1)
        self.params = [self.Wh, self.Wx, self.b, self.U, self.c]

def forward_chunk(net, X, h0):
    B, K, _ = X.shape; m = net.b.size
    H = np.zeros((B, K+1, m)); H[:, 0] = h0
    Z = np.zeros((B, K, m))
    for t in range(K):
        Z[:, t] = H[:, t] @ net.Wh + X[:, t] @ net.Wx + net.b
        H[:, t+1] = np.tanh(Z[:, t])
    return H, Z

def backward_chunk(net, X, H, Z, dH_last):
    # dH_last: (B,m) — 창의 마지막 상태에 대한 기울기 (독립 판독이 마지막에만 있을 때)
    B = X.shape[0]
    gWh = np.zeros_like(net.Wh); gWx = np.zeros_like(net.Wx); gb = np.zeros_like(net.b)
    delta = dH_last
    for t in range(X.shape[1]-1, -1, -1):
        dz = delta * (1 - np.tanh(Z[:, t])**2)
        gWh += H[:, t].T @ dz
        gWx += X[:, t].T @ dz
        gb += dz.sum(axis=0)
        delta = dz @ net.Wh.T
    return gWh, gWx, gb

def adam(p, g, m, v, t, lr=3e-3):
    m[:] = 0.9*m + 0.1*g; v[:] = 0.999*v + 0.001*g*g
    p -= lr*(m/(1-0.9**t))/(np.sqrt(v/(1-0.999**t))+1e-8)

def train_tbptt(delta, K, steps=None, seed=0, track=False):
    steps = steps or (150 if FAST else 350)
    net = RNN(np.random.default_rng(seed))
    ms = [np.zeros_like(p) for p in net.params]; vs = [np.zeros_like(p) for p in net.params]
    rb = np.random.default_rng(500+seed); B = 96
    Xev, yev = make_adding(1000, delta, np.random.default_rng(SEED+77))
    hist = []
    for t in range(1, steps+1):
        X, y = make_adding(B, delta, rb)
        # 순전파: 창 단위로, 상태는 이어 받고 기울기는 창 안에서만
        h = np.zeros((B, M))
        caches = []
        rem = T_SEQ % K
        starts = ([0] if rem else []) + list(range(rem, T_SEQ, K))
        for s0 in starts:                            # 마지막 창의 길이가 정확히 K가 되도록 경계를 뒤에서 맞춘다
            s1 = s0 + (rem if (s0 == 0 and rem) else K)
            Xc = X[:, s0:s1]
            H, Z = forward_chunk(net, Xc, h)
            caches.append((Xc, H, Z, s0))
            h = H[:, -1]                            # detach: 값만 전달
        pred = h @ net.U + net.c
        r = (pred.ravel() - y)
        # 역전파: 마지막 창만 판독 기울기를 받는다 (그 앞 창들은 기울기가 끊겨 0)
        gU = h.T @ (2*r/B); gc = np.array([2*r.mean()])
        dH_last = np.outer(2*r/B, net.U)
        Xc, H, Z, s0 = caches[-1]
        gWh, gWx, gb = backward_chunk(net, Xc, H, Z, dH_last)
        for pp, g, m_, v_ in zip(net.params, [gWh, gWx, gb, gU, gc], ms, vs):
            adam(pp, g, m_, v_, t)
        if track and (t % 10 == 0 or t == 1):
            hp, _ = full_forward(net, Xev)
            mse = np.mean((hp - yev)**2)
            hist.append((t, mse))
    hp, _ = full_forward(net, Xev)
    return np.mean((hp - yev)**2), hist

def full_forward(net, X):
    h = np.zeros((X.shape[0], M))
    for t in range(X.shape[1]):
        h = np.tanh(h @ net.Wh + X[:, t] @ net.Wx + net.b)
    return (h @ net.U + net.c).ravel(), h

# 기준선: 항상 두 표시 값의 합의 평균(=1)을 답하는 상수 예측의 MSE
Xb, yb = make_adding(4000, 16, np.random.default_rng(1))
BASE = np.var(yb)
print(f"상수 예측 기준선 MSE = {BASE:.4f}")

---
## 3. $K$와 $\Delta$를 훑는다

In [ ]:
DELTAS = [8, 16] if FAST else [4, 8, 16]
KS     = [6, 40] if FAST else [3, 6, 12, 24, 40]
SEEDS  = 1 if FAST else 2
mse = np.zeros((len(KS), len(DELTAS), SEEDS))
for ki, K in enumerate(KS):
    for di, d in enumerate(DELTAS):
        for si in range(SEEDS):
            mse[ki, di, si], _ = train_tbptt(d, K, seed=si)
        print(f"K={K:2d}  Δ={d:2d}  MSE={mse[ki, di].mean():.4f}  ({time.time()-_t0:.0f}초)")

# 학습 곡선 두 개 (성공/실패 사례)
_, hist_ok = train_tbptt(16, 40, seed=0, track=True)
_, hist_bad = train_tbptt(16, 10, seed=0, track=True)

---
## 4. 교재 그림 — fig_12_2_5

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) 과제 구조
ax = axes[0]
Xs, ys = make_adding(1, 16, np.random.default_rng(5))
ax.bar(range(T_SEQ), Xs[0, :, 0], color=CB[0], alpha=0.4, label=lab('값 $v_t$', 'values'))
mk = np.where(Xs[0, :, 1] > 0)[0]
ax.bar(mk, Xs[0, mk, 0], color=CB[4], label=lab('표시된 값', 'marked'))
ax.annotate('', xy=(mk[0], 1.06), xytext=(T_SEQ-1, 1.06),
            arrowprops=dict(arrowstyle='<->', color=CB[5]))
ax.text((mk[0]+T_SEQ-1)/2, 1.1, lab('$\\Delta$ = 운반 거리', '$\\Delta$'), ha='center', color=CB[5], fontsize=9)
ax.set_ylim(0, 1.25)
ax.set_xlabel(lab('시각 $t$', 'time $t$')); ax.set_ylabel(lab('값', 'value'))
ax.set_title(lab('(a) 덧셈 과제 — 두 표시 값의 합', '(a) adding problem'), fontsize=10)
ax.legend(fontsize=8)

# (b) K에 따른 최종 MSE (Δ 고정 대표)
ax = axes[1]
di_show = DELTAS.index(16) if 16 in DELTAS else len(DELTAS)-1
m_ = mse[:, di_show].mean(axis=1); s_ = mse[:, di_show].std(axis=1)
ax.errorbar(KS, m_, yerr=s_, fmt='o-', color=CB[5], ms=5, capsize=3)
ax.axhline(BASE, color='k', lw=0.8, ls=':')
ax.text(KS[0], BASE*1.05, lab('상수 예측 수준', 'constant predictor'), fontsize=8)
ax.axvline(DELTAS[di_show], color=CB[4], lw=0.8, ls='--')
ax.text(DELTAS[di_show]*1.02, m_.max()*0.7, lab(f'$\\Delta={DELTAS[di_show]}$', ''), color=CB[4], fontsize=9)
ax.set_xlabel(lab('절단 길이 $K$', 'truncation length $K$'))
ax.set_ylabel(lab('시험 MSE', 'test MSE'))
ax.set_title(lab(f'(b) $\\Delta={DELTAS[di_show]}$에서 절단 길이를 훑으면', '(b) sweep $K$'), fontsize=10)

# (c) 성공 지도
ax = axes[2]
success = (mse.mean(axis=2) < 0.1*BASE).astype(float)
imv = ax.imshow(success, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto'); ax.grid(False)
ax.set_xticks(range(len(DELTAS))); ax.set_xticklabels(DELTAS)
ax.set_yticks(range(len(KS))); ax.set_yticklabels(KS)
for ki in range(len(KS)):
    for di in range(len(DELTAS)):
        ax.text(di, ki, lab('성공', 'ok') if success[ki, di] else lab('실패', 'fail'),
                ha='center', va='center', fontsize=8)
ax.set_xlabel(lab('의존 거리 $\\Delta$', 'dependency $\\Delta$'))
ax.set_ylabel(lab('절단 길이 $K$', 'truncation $K$'))
ax.set_title(lab('(c) 성공 지도 — 경계는 $K=\\Delta$ 근처', '(c) success map'), fontsize=10)

# (d) 학습 곡선: 조용한 실패
ax = axes[3]
ax.plot([h[0] for h in hist_ok], [h[1] for h in hist_ok], '-', color=CB[5], lw=1.4,
        label=lab('$K=40>\\Delta$ (성공)', '$K=40$'))
ax.plot([h[0] for h in hist_bad], [h[1] for h in hist_bad], '-', color=CB[4], lw=1.4,
        label=lab('$K=10<\\Delta$ (실패)', '$K=10$'))
ax.axhline(BASE, color='k', lw=0.8, ls=':')
ax.set_xlabel(lab('학습 걸음', 'step')); ax.set_ylabel(lab('시험 MSE', 'test MSE'))
ax.set_title(lab('(d) 실패는 요동이 아니라 조용한 평평함이다', '(d) failure is quiet'), fontsize=10)
ax.legend(fontsize=8)

save_book_fig(fig, 'fig_12_2_5')
plt.show()

> ### 읽는 법
>
> (b, c) 성공과 실패의 경계가 $K\approx\Delta$에서 계단으로 갈린다 — 명제 12.2.1의 예측 그대로다.
> 정밀하게 보면 경계에 두어 스텝의 여유가 있다. 기울기가 끊겨도 **상태는 몇 스텝쯤 정보를 실어 나르고**
> (§12.2.3의 "실려 오는 정보"; 무작위 순환의 저수지 효과), 판독기가 그것을 주워 쓰기 때문이다.
> 실패 칸의 MSE가 기준선보다 약간 낮은 것도 같은 이유다. 정보가 실려 오는 것과 그것을 잘 싣도록
> $W_h$가 학습되는 것은 다르다는 구분이 숫자로 보인다.
> (d) 실패 곡선은 요동치지 않는다. **지역 정보로 낼 수 있는 성능까지 내려간 뒤 조용히 평평하다.**
> 학습이 덜 된 모양이 아니므로, 절단이 원인이라는 의심 자체가 들지 않기 쉽다 (§12.2.6).
>
> ⚠︎ $\Delta$를 24 이상으로 키우면 $K=40$(절단 없음)에서도 실패한다. 절단이 아니라 §12.3의
> 기울기 소실이 병목이 되는 체제다. 두 상한(절단, 소실)은 별개이며, 이 실험의 $\Delta$ 범위는
> 소실 상한 아래에서 절단 상한만 보이도록 고른 것이다. 소실 쪽은 §12.3.5와 §12.4.5에서 다룬다.

---
## 5. 자기 점검

1. 실패 사례의 MSE가 정확히 기준선이 아니라 그보다 조금 낮은 이유는? (힌트: 두 표시 중 하나는 창 안에 있다)
2. 이 실험의 판독은 마지막 상태 하나에서만 이루어진다. 매 시각 판독이 있는 과제라면 절단의 피해가 어떻게 달라지는가?
3. $K=\Delta$ 정확히 그 지점의 성패는 창 경계의 위상에 달렸다. 왜인가?
4. 상태를 창마다 0으로 초기화하는 변형(상태도 끊기)을 실행하면 (c)의 경계가 어떻게 움직이는가?

## 6. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `T_SEQ` | 1절 | 40 | 시퀀스 길이 |
| `DELTAS`, `KS` | 3절 | — | 지도의 해상도 |
| 성공 판정 계수 | 4절 | 0.1 | 기준선 대비 성공 문턱 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")